# Pulse calibration workflow (qubex-emulator)

> This is the qubex-emulator version of the upstream qubex experiment notebook. It uses synthetic `FakeExperiment` results and does not connect to hardware.

This notebook calibrates the default half-pi / pi pulses first, then
shows how to refine the DRAG versions. The goal is to leave you with
inspectable cached pulses and quick validation plots.

## 1. Create an `Experiment`

Select one qubit and load the configuration that contains its current
pulse parameters.

In [1]:
import numpy as np

from IPython.display import display

import qubex_emulator as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

## 2. Connect to the setup

Connect before you run any checks or pulse calibrations.

In [2]:
exp.connect()

FakeExperiment(name='fake-qubex-two-qubit-system', device_id='YOUR_SYSTEM_ID', qubit_labels=('Q00', 'Q01', 'Q02', 'Q03'), qubit_frequencies=(7.157231, 8.032295, 7.812112, 6.944337), qubit_anharmonicities=(-0.393715, -0.487412, -0.421337, -0.365884), readout_frequencies=(6.752, 6.903, 6.844, 6.711), coupling_strength=0.005, qubit_lifetime=(20.0, 20.0), qubit_lifetimes=None, hpi_duration=24.0, pi_duration=24.0, readout_duration=1000.0, rzx90_duration=None, cx_duration=None, single_qubit_fidelity=None, two_qubit_fidelity=None, readout_assignment_error=None, positions=((0.0, 0.0), (1.0, 0.0), (2.0, 0.0), (3.0, 0.0)), calibrated_at=None, metadata={'chip_id': None, 'system_id': 'YOUR_SYSTEM_ID', 'muxes': [0], 'calib_note_path': None, 'calibration_valid_days': None, 'extra_options': {}}, readout_pre_margin=0.0, readout_post_margin=0.0, config_path='', params_path='', property_dir=PosixPath('.'), classifier_dir=PosixPath('.'), classifier_type='gmm', configuration_mode='ge-cr-cr', drag_hpi_puls

## 3. Check the waveform

Start with a waveform check so you can catch obvious readout issues before tuning pulse amplitudes.

In [3]:
waveform_result = exp.check_waveform()

## 4. Measure a baseline Rabi oscillation

Use the initial Rabi measurement as a reference before you update the pulse amplitude.

In [4]:
rabi_result = exp.obtain_rabi_params()

## 5. Calibrate the default half-pi pulse

Run the half-pi calibration first because many later workflows depend on this pulse.

In [5]:
hpi_result = exp.calibrate_hpi_pulse(
    targets=exp.qubit_labels,
    n_rotations=1,
)

## 6. Reload and re-check the Rabi oscillation

After updating the control amplitude file, reload and confirm that the Rabi trace reflects the new value.

In [6]:
# Update `control_amplitude.yaml` manually, then reload
exp.reload()

# Re-check the Rabi oscillation
updated_rabi_result = exp.obtain_rabi_params()

## 7. Plot the calibrated pulse

Plot the cached half-pi pulse for the target qubit so you can inspect the waveform directly.

In [7]:
Q0 = exp.qubit_labels[0]

hpi_pulse = exp.hpi_pulse[Q0]
display(hpi_pulse.plot())

None

## 8. Repeat the calibrated half-pi pulse

Use a repeated-sequence check to confirm that the calibrated pulse accumulates as expected.

In [8]:
repeat_result = exp.repeat_sequence(
    {Q0: hpi_pulse},
    repetitions=20,
)

## 9. Optionally save the plot

Export the repeated-sequence plot only when you want to keep a record of the result.

In [9]:
display(repeat_result.plot(normalize=True, images_dir="./images"))

None